In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from matplotlib.colors import LogNorm
from pathlib import Path
import geopandas as gp
import spatialdata as sd
from tqdm import tqdm
from shapely.geometry import box
import geopandas as gpd
from spatialdata.models import ShapesModel, Labels2DModel, Image2DModel
from src.utils import read_obs
from datetime import datetime

/opt/conda/lib/python3.12/importlib/__init__.py:90: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  return _bootstrap._gcd_import(name[level:], package, level)


In [2]:
current_datetime = datetime.now()
formatted_date = current_datetime.strftime("%Y-%m-%d")

In [3]:
dfs = {"cell_dfs":{}, "tau_dfs":{}, "plaque_dfs":{}}

# pTau DataFrame

In [4]:
sdata_paths = list(Path("/data/sdata_ptau_1").glob("*.zarr")) + list(Path("/data/sdata_ptau_2").glob("*.zarr")) 

for path in sdata_paths:
    barcode = path.stem.split("_")[0]
    sdata = sd.read_zarr(path)
    dfs['tau_dfs'][barcode] = sd.transform(sdata['cellular_tau_boundaries'], to_coordinate_system='global')

/tmp/ipykernel_75967/909848288.py:5: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata = sd.read_zarr(path)
2026-02-04 00:27:26 | [INFO] root_attr: multiscales
2026-02-04 00:27:26 | [INFO] root_attr: omero
2026-02-04 00:27:26 | [INFO] root_attr: spatialdata_attrs
2026-02-04 00:27:26 | [INFO] datasets [{'coordinateTransformations': [{'scale': [1.0, 1.0, 1.0], 'type': 'scale'}], 'path': '0'}, {'coordinateTransformations': [{'scale': [1.0, 2.000039878768544, 2.0], 'type': 'scale'}], 'path': '1'}, {'coordinateTransformations': [{'scale': [1.0, 4.000079757537088, 4.0], 'type': 'scale'}], 'path': '2'}, {'coordinateTransformations': [{'scale': [1.0, 8.000159515074175, 8.0], 'type': 'scale'}], 'path': '3'}, {'coordinateTransformations': [{'scale': [1.0, 16.002871729419272, 16.0], 'type': 'scale'}], 'path': '4'}]
2026-02-04 00:27:26 | [INFO] resolution: 0
2026-02-04 00:27:26 | [I

In [5]:
tau_df = pd.concat(dfs['tau_dfs']).drop('label', axis = 1).reset_index().rename(columns = {"level_0":"barcode"})
# plaque_df = pd.concat(dfs['plaque_dfs']).drop('label', axis = 1).reset_index().rename(columns = {"level_0":"barcode"})

In [6]:
tau_df.to_csv(f"/results/seaad_cah_tau_polygons.{formatted_date}.csv")

# Plaque DataFrame

In [7]:
neuropath_obs = pd.read_csv("/root/capsule/data/Supplemental Tables/Supplemental Table 7.csv", index_col = 0)

In [8]:
high_plaque_donors = neuropath_obs[neuropath_obs['percent AT8 positive area'] > 0.05].index

In [9]:
obs = read_obs("/root/capsule/data/combined_adata/CaH_Xenium.2026-01-07.h5ad")

In [10]:
section_barcodes = obs[(obs['Donor ID'].isin(high_plaque_donors)) & (obs['Used in analysis'])]['barcode'].unique()

In [11]:
for barcode in section_barcodes:
    path = Path(f"/root/capsule/data/spatialdata/{barcode}_with_vs200_CAH_processed.zarr")
    sdata = sd.read_zarr(path)
    dfs['plaque_dfs'][barcode] = sd.transform(sdata['plaque_boundaries'], to_coordinate_system='global')

/tmp/ipykernel_75967/794367380.py:3: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata = sd.read_zarr(path)
2026-02-04 00:28:45 | [INFO] root_attr: multiscales
2026-02-04 00:28:45 | [INFO] root_attr: omero
2026-02-04 00:28:45 | [INFO] root_attr: spatialdata_attrs
2026-02-04 00:28:45 | [INFO] datasets [{'coordinateTransformations': [{'scale': [1.0, 1.0, 1.0], 'type': 'scale'}], 'path': '0'}, {'coordinateTransformations': [{'scale': [1.0, 2.000132485426603, 2.0000638202820857], 'type': 'scale'}], 'path': '1'}, {'coordinateTransformations': [{'scale': [1.0, 3.99973506424692, 4.000382946132244], 'type': 'scale'}], 'path': '2'}, {'coordinateTransformations': [{'scale': [1.0, 8.002650410813676, 8.000765892264488], 'type': 'scale'}], 'path': '3'}, {'coordinateTransformations': [{'scale': [1.0, 16.009544008483562, 16.00561797752809], 'type': 'scale'}], 'path': '4'}]
2026-02-04 0

In [12]:
plaque_df = pd.concat(dfs['plaque_dfs']).drop('label', axis = 1).reset_index().rename(columns = {"level_0":"barcode"})

In [13]:
plaque_df.to_csv(f"/results/seaad_cah_plaque_polygons.{formatted_date}.csv")

# Cell DataFrame

In [14]:
section_barcodes = obs[obs['Used in analysis']]['barcode'].unique()

In [15]:
for barcode in section_barcodes:
    path = Path(f"/root/capsule/data/spatialdata/{barcode}_with_vs200_CAH_processed.zarr")
    sdata = sd.read_zarr(path)
    dfs['cell_dfs'][barcode] = sd.transform(sdata['cell_boundaries'], to_coordinate_system='global')

/tmp/ipykernel_75967/1040669870.py:3: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata = sd.read_zarr(path)
2026-02-04 00:30:39 | [INFO] root_attr: multiscales
2026-02-04 00:30:39 | [INFO] root_attr: omero
2026-02-04 00:30:39 | [INFO] root_attr: spatialdata_attrs
2026-02-04 00:30:39 | [INFO] datasets [{'coordinateTransformations': [{'scale': [1.0, 1.0, 1.0], 'type': 'scale'}], 'path': '0'}, {'coordinateTransformations': [{'scale': [1.0, 2.0, 2.0000638202820857], 'type': 'scale'}], 'path': '1'}, {'coordinateTransformations': [{'scale': [1.0, 4.000264970853206, 4.000382946132244], 'type': 'scale'}], 'path': '2'}, {'coordinateTransformations': [{'scale': [1.0, 8.000529941706413, 8.000765892264488], 'type': 'scale'}], 'path': '3'}, {'coordinateTransformations': [{'scale': [1.0, 16.001059883412825, 16.00561797752809], 'type': 'scale'}], 'path': '4'}]
2026-02-04 00:30:39 | [I

In [16]:
cell_df = pd.concat(dfs['cell_dfs']).reset_index().rename(columns = {'level_0':'barcode', 'level_1':'label'})

In [17]:
cell_df.index = cell_df['label'] + "_" + cell_df['barcode']

In [18]:
obs.index = obs['cell_id'].astype(str) + "_" + obs['barcode'].astype(str)

In [19]:
cluster_columns = ['Neighborhood', 'Subclass', 'Supertype']

In [20]:
cell_df = cell_df.merge(obs[cluster_columns], left_index = True, right_index = True, how = "left")

In [21]:
cell_df.to_csv(f"/results/seaad_cah_cell_segmentation_polgyons.{formatted_date}.csv")